In [ ]:
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T
import segmentation_models_pytorch as smp
from transformers import TrainingArguments, Trainer
from transformers.modeling_outputs import SemanticSegmenterOutput

# ==========================================
# 1. AYARLAR VE PATH TANIMLAMALARI
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Eğitim seti klasörleri (SegFormer'da kullandığın ayrılmış eğitim resimleri)
IMAGE_DIR = "/content/drive/MyDrive/TrainingData/Imgdir"
MASK_DIR = "/content/drive/MyDrive/TrainingData/MaskDir"

# DeepLab sonuçlarının kaydedileceği yeni klasör
OUTPUT_DIR = "/content/drive/MyDrive/deeplabv3-paintings"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{OUTPUT_DIR}/runs"

COLOR_MAP = {
    (0, 0, 0): 0,          # background
    (128, 0, 0): 1,        # building
    (0, 128, 0): 2,        # earth
    (128, 128, 0): 3,      # grass
    (0, 0, 128): 4,        # animal
    (128, 0, 128): 5,      # mountain
    (0, 128, 128): 6,      # path_road
    (128, 128, 128): 7,    # person
    (64, 0, 0): 8,         # rock
    (192, 0, 0): 9,        # shrub_bush
    (64, 128, 0): 10,      # sky
    (192, 128, 0): 11,     # tree_conical
    (64, 0, 128): 12,      # tree_broadleaf
    (192, 0, 128): 13,     # wooded_mass
    (64, 128, 128): 14     # Water
}
num_classes = 15

def rgb_to_id(mask_pil):
    mask_np = np.array(mask_pil.convert("RGB"))
    h, w, _ = mask_np.shape
    id_mask = np.zeros((h, w), dtype=np.int64)
    for rgb, idx in COLOR_MAP.items():
        if idx == 0: continue
        match = (mask_np[:, :, 0] == rgb[0]) & (mask_np[:, :, 1] == rgb[1]) & (mask_np[:, :, 2] == rgb[2])
        id_mask[match] = idx
    return id_mask

# ==========================================
# 2. DEEPLAB İÇİN DATASET SINIFI (Manuel İşlemci)
# ==========================================
# İşlemci (Processor) sınıfı HF'de olmadığı için veriyi manuel hazırlıyoruz
class DeepLabDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.img_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith('.png')])

        # DeepLab için standart ImageNet normalizasyonu
        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(IMAGE_DIR, self.img_files[idx])
        mask_path = os.path.join(MASK_DIR, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        color_mask = Image.open(mask_path)

        # Modelin hafızaya sığması için standart bölütleme çözünürlüğüne (512x512) ölçekleme
        image = image.resize((512, 512), Image.BILINEAR)
        color_mask = color_mask.resize((512, 512), Image.NEAREST)

        id_mask_np = rgb_to_id(color_mask)

        image_tensor = self.transform(image)
        labels_tensor = torch.tensor(id_mask_np, dtype=torch.long)

        return {
            "pixel_values": image_tensor,
            "labels": labels_tensor
        }

# ==========================================
# 3. HUGGING FACE TRAINER İÇİN DEEPLABV3+ SARMALAYICISI (WRAPPER)
# ==========================================
class DeepLabV3PlusHuggingFace(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # SMP kütüphanesinden ResNet-50 omurgalı DeepLabV3+ çağırıyoruz
        self.model = smp.DeepLabV3Plus(
            encoder_name="resnet50",        # En sağlam ve dengeli CNN omurgası
            encoder_weights="imagenet",     # Zengin ön-eğitim
            in_channels=3,
            classes=num_classes
        )
        # Background (0) sınıfını hata (loss) hesaplamasına katmıyoruz
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    def forward(self, pixel_values, labels=None):
        logits = self.model(pixel_values)
        loss = None

        if labels is not None:
            loss = self.loss_fn(logits, labels)

        # Hugging Face Trainer'ın anlayacağı standart çıktı formatı
        return SemanticSegmenterOutput(loss=loss, logits=logits)

# Modeli ve Veri Setini Başlat
model = DeepLabV3PlusHuggingFace(num_classes=num_classes).to(device)
train_dataset = DeepLabDataset(IMAGE_DIR, MASK_DIR)

# ==========================================
# 4. EĞİTİM (TRAINING) ARGÜMANLARI VE BAŞLATMA
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-4,              # CNN'ler Transformer'lara göre biraz daha büyük LR sever
    num_train_epochs=100,            # Diğer modellerle eşit şartlarda yarışması için 100 epoch
    per_device_train_batch_size=2,
    save_strategy="epoch",
    logging_steps=5,
    remove_unused_columns=False,
    report_to="tensorboard"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting DeepLabV3+ (ResNet-50) training loop...")
trainer.train()

# Eğitilen model ağırlıklarını kaydet (PyTorch Standart Formatı)
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "deeplabv3_custom_weights.pth"))
print(f"[BAŞARILI] DeepLabV3+ ağırlıkları Drive'a kaydedildi: {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Starting DeepLabV3+ (ResNet-50) training loop...


Step,Training Loss
5,2.785741
10,2.461762
15,2.116294
20,1.813905
25,1.575484
30,1.351279
35,1.237925
40,1.113138
45,1.021379
50,0.962734


[BAŞARILI] DeepLabV3+ ağırlıkları Drive'a kaydedildi: /content/drive/MyDrive/deeplabv3-paintings


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import torch.nn.functional as F
import segmentation_models_pytorch as smp

# ==========================================
# 1. AYARLAR VE PATH TANIMLAMALARI
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_TYPE = "deeplabv3plus"

# Kaydettiğin DeepLab modelinin ve Test verilerinin yolları
MODEL_DIR = "/content/drive/MyDrive/deeplabv3-paintings"
WEIGHTS_PATH = os.path.join(MODEL_DIR, "deeplabv3_custom_weights.pth")

TEST_IMAGE_DIR = "/content/drive/MyDrive/TestData/TestImages"
TEST_MASK_DIR = "/content/drive/MyDrive/TestData/TestMasks"

# Sonuçların kaydedileceği klasör
RESULTS_DIR = os.path.join(MODEL_DIR, "Test_Results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ==========================================
# 2. YENİ RENK HARİTASI (CVAT TEST SETİ RENKLERİ)
# ==========================================
COLOR_MAP = {
    (0, 0, 0): 0,          # background
    (110, 13, 13): 1,      # building
    (96, 66, 7): 2,        # earth
    (131, 224, 112): 3,    # grass
    (240, 120, 240): 4,    # animal
    (37, 70, 103): 5,      # mountain
    (230, 209, 168): 6,    # path_road
    (184, 61, 245): 7,     # person
    (65, 93, 125): 8,      # rock
    (48, 173, 48): 9,      # shrub_bush
    (18, 206, 242): 10,    # sky
    (13, 135, 53): 11,     # tree_conical
    (135, 246, 171): 12,   # tree_broadleaf
    (253, 164, 5): 13,     # wooded_mass
    (85, 144, 203): 14     # Water
}

class_names = [
    "background", "building", "earth", "grass", "animal",
    "mountain", "path_road", "person", "rock", "shrub_bush",
    "sky", "tree_conical", "tree_broadleaf", "wooded_mass", "Water"
]
num_classes = len(class_names)

def rgb_to_id(mask_pil):
    mask_np = np.array(mask_pil.convert("RGB"))
    h, w, _ = mask_np.shape
    id_mask = np.zeros((h, w), dtype=np.int64)
    for rgb, idx in COLOR_MAP.items():
        if idx == 0: continue
        match = (mask_np[:, :, 0] == rgb[0]) & (mask_np[:, :, 1] == rgb[1]) & (mask_np[:, :, 2] == rgb[2])
        id_mask[match] = idx
    return id_mask

# ==========================================
# 3. MODELİ VE AĞIRLIKLARI YÜKLE
# ==========================================
class DeepLabV3PlusHuggingFace(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights=None, # Eval aşamasında indiremeye gerek yok
            in_channels=3,
            classes=num_classes
        )
    def forward(self, pixel_values):
        return self.model(pixel_values)

print(f"\n[DEEPLABV3+] Modeli test için Drive'dan yükleniyor...")
model = DeepLabV3PlusHuggingFace(num_classes=num_classes).to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
model.eval()

# Görüntüleri modelin istediği formata getiren dönüşüm (Transform)
transform = T.Compose([
    T.Resize((512, 512)), # Eğitimdeki boyut
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 4. ÇIKARIM VE KARIŞIKLIK MATRİSİ (INFERENCE)
# ==========================================
img_files = sorted([f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
mask_files = sorted([f for f in os.listdir(TEST_MASK_DIR) if f.lower().endswith('.png')])

assert len(img_files) == len(mask_files), "HATA: Resim ve Maske sayıları eşit değil!"

confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

print(f"{len(img_files)} adet resim üzerinde piksel piksel test yapılıyor. Lütfen bekleyin...")

for img_name, mask_name in zip(img_files, mask_files):
    # Orijinal resim ve maske
    original_image = Image.open(os.path.join(TEST_IMAGE_DIR, img_name)).convert("RGB")
    gt_id_mask = rgb_to_id(Image.open(os.path.join(TEST_MASK_DIR, mask_name)))

    # Model girdisini hazırla
    input_tensor = transform(original_image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)

    # Çıktıyı 512x512'den Orijinal Resim boyutuna geri büyüt
    upsampled_logits = F.interpolate(
        logits, size=original_image.size[::-1], mode="bilinear", align_corners=False
    )
    pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy()

    flat_gt = gt_id_mask.flatten()
    flat_pred = pred_mask.flatten()
    indices = (flat_gt >= 0) & (flat_gt < num_classes)

    confusion_matrix += np.bincount(
        num_classes * flat_gt[indices] + flat_pred[indices],
        minlength=num_classes**2
    ).reshape(num_classes, num_classes)

# ==========================================
# 5. METRİKLERİ HESAPLA VE KAYDET
# ==========================================
print("\n" + "="*60 + f"\nDEEPLABV3+ (RESNET-50) - METRİK HESAPLAMALARI\n" + "="*60)

tp = np.diag(confusion_matrix)
fp = np.sum(confusion_matrix, axis=0) - tp
fn = np.sum(confusion_matrix, axis=1) - tp
eps = 1e-15

iou_per_class = tp / (tp + fp + fn + eps)
precision_per_class = tp / (tp + fp + eps)
recall_per_class = tp / (tp + fn + eps)

total_pixels_per_class = np.sum(confusion_matrix, axis=1)
total_valid_pixels = np.sum(total_pixels_per_class)
class_frequencies = total_pixels_per_class / (total_valid_pixels + eps)
fwiou = np.sum(class_frequencies * iou_per_class)

results_data = []
present_classes_iou = []

for i, name in enumerate(class_names):
    if total_pixels_per_class[i] > 0:
        iou_val = iou_per_class[i] * 100
        prec_val = precision_per_class[i] * 100
        rec_val = recall_per_class[i] * 100
        present_classes_iou.append(iou_per_class[i])

        print(f"Sınıf: {name:<15} | IoU: %{iou_val:05.2f} | P: %{prec_val:05.2f} | R: %{rec_val:05.2f}")

        results_data.append({
            "Class Name": name,
            "IoU (%)": round(iou_val, 2),
            "Precision (%)": round(prec_val, 2),
            "Recall (%)": round(rec_val, 2)
        })

miou = np.mean(present_classes_iou) * 100
mean_precision = np.mean([precision_per_class[i] for i in range(num_classes) if total_pixels_per_class[i] > 0]) * 100
mean_recall = np.mean([recall_per_class[i] for i in range(num_classes) if total_pixels_per_class[i] > 0]) * 100
fwiou_percent = fwiou * 100

print("-" * 60)
print(f"GENEL mIoU (Mean IoU)                : %{miou:.2f}")
print(f"GENEL FWIoU (Ağırlıklı IoU)          : %{fwiou_percent:.2f}")
print(f"GENEL MEAN PRECISION (Hassasiyet)    : %{mean_precision:.2f}")
print(f"GENEL MEAN RECALL (Duyarlılık)       : %{mean_recall:.2f}")
print("=" * 60)

# CSV DOSYASINA KAYDET
df = pd.DataFrame(results_data)
csv_path = os.path.join(RESULTS_DIR, f"{MODEL_TYPE}_class_metrics.csv")
df.to_csv(csv_path, index=False)

# TXT RAPOR DOSYASINA KAYDET
txt_path = os.path.join(RESULTS_DIR, f"{MODEL_TYPE}_overall_report.txt")
with open(txt_path, "w") as f:
    f.write(f"--- DEEPLABV3+ (ResNet-50) 6-IMAGE TEST REPORT ---\n")
    f.write(f"Mean IoU (mIoU)       : {miou:.2f}%\n")
    f.write(f"Freq Weighted IoU     : {fwiou_percent:.2f}%\n")
    f.write(f"Mean Precision        : {mean_precision:.2f}%\n")
    f.write(f"Mean Recall           : {mean_recall:.2f}%\n")

print(f"\n[BAŞARILI] İşlem tamamlandı! Sonuçlar Drive'a kaydedildi.")


[DEEPLABV3+] Modeli test için Drive'dan yükleniyor...
6 adet resim üzerinde piksel piksel test yapılıyor. Lütfen bekleyin...

DEEPLABV3+ (RESNET-50) - METRİK HESAPLAMALARI
Sınıf: background      | IoU: %00.00 | P: %00.00 | R: %00.00
Sınıf: building        | IoU: %55.03 | P: %78.60 | R: %64.73
Sınıf: earth           | IoU: %29.84 | P: %37.95 | R: %58.26
Sınıf: grass           | IoU: %16.71 | P: %32.83 | R: %25.39
Sınıf: animal          | IoU: %53.92 | P: %59.26 | R: %85.67
Sınıf: mountain        | IoU: %00.00 | P: %00.00 | R: %00.00
Sınıf: path_road       | IoU: %04.95 | P: %23.53 | R: %05.90
Sınıf: person          | IoU: %34.90 | P: %48.42 | R: %55.56
Sınıf: rock            | IoU: %49.20 | P: %53.02 | R: %87.21
Sınıf: shrub_bush      | IoU: %21.26 | P: %26.82 | R: %50.63
Sınıf: sky             | IoU: %94.02 | P: %94.36 | R: %99.61
Sınıf: tree_conical    | IoU: %14.10 | P: %97.21 | R: %14.16
Sınıf: tree_broadleaf  | IoU: %32.85 | P: %32.91 | R: %99.47
Sınıf: wooded_mass     | IoU: %11.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

# ==========================================
# 1. AYARLAR VE PATH TANIMLAMALARI
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modelinin ve ağırlıklarının bulunduğu yol
MODEL_DIR = "/content/drive/MyDrive/deeplabv3-paintings"
WEIGHTS_PATH = os.path.join(MODEL_DIR, "deeplabv3_custom_weights.pth")

# Deneme yapacağın resimlerin klasörü (Etiketsiz ham resimler)
TEST_IMAGE_DIR = "/content/drive/MyDrive/SemanticSegmentationV1/ProjectImages"

# Görsel sonuçların kaydedileceği YENİ KLASÖR
OUTPUT_RESULTS_DIR = os.path.join(MODEL_DIR, "Visual_Inference_Results")
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

# ==========================================
# 2. RENK PALETİ VE SINIF İSİMLERİ (Görselleştirme İçin)
# ==========================================
class_names = [
    "background", "building", "earth", "grass", "animal",
    "mountain", "path_road", "person", "rock", "shrub_bush",
    "sky", "tree_conical", "tree_broadleaf", "wooded_mass", "Water"
]

# Çıktı maskelerini boyamak için standart Pascal VOC Renk Paleti (RGB)
PALETTE = np.array([
    [0, 0, 0],        # 0: background
    [128, 0, 0],      # 1: building
    [0, 128, 0],      # 2: earth
    [128, 128, 0],    # 3: grass
    [0, 0, 128],      # 4: animal
    [128, 0, 128],    # 5: mountain
    [0, 128, 128],    # 6: path_road
    [128, 128, 128],  # 7: person
    [64, 0, 0],       # 8: rock
    [192, 0, 0],      # 9: shrub_bush
    [64, 128, 0],     # 10: sky
    [192, 128, 0],    # 11: tree_conical
    [64, 0, 128],     # 12: tree_broadleaf
    [192, 0, 128],    # 13: wooded_mass
    [64, 128, 128]    # 14: Water
])

# ==========================================
# 3. MODELİ YÜKLEME
# ==========================================
class DeepLabV3PlusHuggingFace(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights=None,
            in_channels=3,
            classes=num_classes
        )
    def forward(self, pixel_values):
        return self.model(pixel_values)

print("DeepLabV3+ modeli test (Inference) için yükleniyor...")
model = DeepLabV3PlusHuggingFace(num_classes=len(class_names)).to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
model.eval()

# Modelin eğitimi sırasında kullandığı transform formatı
transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 4. GÖRSELLERİ İŞLEME VE KAYDETME
# ==========================================
test_images = sorted([f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"\nToplam {len(test_images)} adet resim üzerinde görsel maskeleme başlatılıyor...\n")

for img_name in test_images:
    img_path = os.path.join(TEST_IMAGE_DIR, img_name)
    original_image = Image.open(img_path).convert("RGB")

    # Resmi tensor'a çevir ve modele gönder
    input_tensor = transform(original_image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)

    # DeepLab'ın 512x512'lik tahminini orijinal resim boyutuna geri esnetiyoruz
    upsampled_logits = F.interpolate(
        logits, size=original_image.size[::-1], mode="bilinear", align_corners=False
    )

    # En yüksek ihtimalli sınıfları seç (Piksel ID matrisi)
    pred_seg = upsampled_logits.argmax(dim=1)[0].cpu().numpy()

    # ------------------------------------------
    # SINIF YÜZDELERİNİ HESAPLAMA (Ekrana Bastırma)
    # ------------------------------------------
    total_pixels = pred_seg.size
    unique_ids, counts = np.unique(pred_seg, return_counts=True)
    pixel_counts = dict(zip(unique_ids, counts))

    print("-" * 50)
    print(f"🖼️ DEEPLABV3+ TAHMİNİ: {img_name}")

    for class_id in range(len(class_names)):
        count = pixel_counts.get(class_id, 0)
        percentage = (count / total_pixels) * 100
        if percentage > 0.0:
            print(f"  * {class_names[class_id]:<15}: %{percentage:.2f}")

    # ------------------------------------------
    # RENKLİ MASKE VE OVERLAY OLUŞTURMA
    # ------------------------------------------
    # Tahmin ID'lerini renk paletiyle boya
    color_seg = np.zeros((pred_seg.shape[0], pred_seg.shape[1], 3), dtype=np.uint8)
    for label, color in enumerate(PALETTE):
        color_seg[pred_seg == label] = color

    # Orijinal resim ile maskeyi %50/%50 şeffaflıkla harmanla
    img_np = np.array(original_image)
    overlay = (img_np * 0.5 + color_seg * 0.5).astype(np.uint8)

    # Yan yana tek bir görselde birleştir
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(original_image)
    axes[0].set_title("Orijinal Tablo", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title("DeepLabV3+ Tahmini", fontsize=14)
    axes[1].axis("off")

    # Çıktıyı dosyaya kaydet ve arabelleği temizle
    save_path = os.path.join(OUTPUT_RESULTS_DIR, f"deeplab_res_{img_name}")
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.close()

print(f"\n[BAŞARILI] İşlem bitti! Tüm maskeli görseller şu klasöre kaydedildi:")
print(f"👉 {OUTPUT_RESULTS_DIR}")

DeepLabV3+ modeli test (Inference) için yükleniyor...

Toplam 20 adet resim üzerinde görsel maskeleme başlatılıyor...

--------------------------------------------------
🖼️ DEEPLABV3+ TAHMİNİ: N-0109-00-000032-wpu.jpg
  * earth          : %3.75
  * grass          : %3.49
  * animal         : %6.07
  * path_road      : %0.64
  * person         : %0.26
  * rock           : %1.15
  * shrub_bush     : %10.80
  * sky            : %12.32
  * tree_broadleaf : %60.30
  * Water          : %1.22
--------------------------------------------------
🖼️ DEEPLABV3+ TAHMİNİ: N-0134-00-000011-wpu.jpg
  * building       : %9.75
  * earth          : %9.50
  * grass          : %7.66
  * path_road      : %0.03
  * person         : %0.60
  * rock           : %0.82
  * shrub_bush     : %5.33
  * sky            : %25.12
  * tree_broadleaf : %39.97
  * Water          : %1.23
--------------------------------------------------
🖼️ DEEPLABV3+ TAHMİNİ: N-0152-00-000014-wpu.jpg
  * building       : %0.27
  * earth   